# 06 — Gold V2 e OneLake Power BI

Reconstrução do fluxo V2 discutido na conversa. Leia README; execute somente após reconciliar Silver. **Publicação multi-tabela não é atômica**.

In [ ]:
from pyspark.sql import functions as F
from pathlib import Path
from uuid import uuid4
from datetime import datetime,timezone
import json
EXECUTION_ID=str(uuid4())
ROOT=Path('/lakehouse/default/Files/media')
assert ROOT.exists(),'Lakehouse padrao nao disponivel'
PUBLICAR_GOLD=False # True apenas apos checar origem e reconciliacao
CONFIRMO_LAKEHOUSE_PADRAO=False
PERMITIR_PUBLICACAO_PARCIAL=True
SILVER={'SIH':'dbo.silver_v2_sih_internacoes','SIM':'dbo.silver_v2_sim_obitos','CNES_ST':'dbo.silver_v2_cnes_estabelecimentos','CNES_LT':'dbo.silver_v2_cnes_leitos'}
GOLD={'CID':'dbo.gold_v2_dim_cid','TEMPO':'dbo.gold_v2_dim_tempo','INTERNACOES':'dbo.gold_v2_fato_internacoes','MORTALIDADE':'dbo.gold_v2_fato_mortalidade','ESTABELECIMENTOS':'dbo.gold_v2_fato_cnes_estabelecimentos','LEITOS':'dbo.gold_v2_fato_cnes_leitos','COBERTURA':'dbo.gold_v2_controle_cobertura','INDICADORES':'dbo.gold_v2_indicadores_uf_ano'}
def tem(t):return spark.catalog.tableExists(t)
for k in ['SIH','SIM']:
 if not tem(SILVER[k]):raise RuntimeError('Silver obrigatoria ausente: '+SILVER[k])
sih=spark.table(SILVER['SIH']);sim=spark.table(SILVER['SIM'])
st=spark.table(SILVER['CNES_ST']) if tem(SILVER['CNES_ST']) else None
lt=spark.table(SILVER['CNES_LT']) if tem(SILVER['CNES_LT']) else None
assert not sim.filter(F.col('uf_arquivo')=='BR').limit(1).count(),'SIM BR pode duplicar estaduais'


In [ ]:
UF_CODIGOS={'11':'RO','12':'AC','13':'AM','14':'RR','15':'PA','16':'AP','17':'TO','21':'MA','22':'PI','23':'CE','24':'RN','25':'PB','26':'PE','27':'AL','28':'SE','29':'BA','31':'MG','32':'ES','33':'RJ','35':'SP','41':'PR','42':'SC','43':'RS','50':'MS','51':'MT','52':'GO','53':'DF'}
UF_EXPR=F.create_map(*[e for k,v in UF_CODIGOS.items() for e in (F.lit(k),F.lit(v))])
def prep(df,municipio,diagnostico,ano,mes):
 digitos=F.regexp_replace(F.col(municipio).cast('string'),r'[^0-9]','')
 d=df.withColumn('municipio_residencia_codigo',F.when(F.length(digitos).isin(6,7),F.substring(digitos,1,6)))
 d=d.withColumn('uf_residencia',F.element_at(UF_EXPR,F.substring('municipio_residencia_codigo',1,2))).withColumn('codigo_uf_ibge',F.substring('municipio_residencia_codigo',1,2).cast('int'))
 d=d.withColumn('ano',ano.cast('int')).withColumn('mes',mes.cast('int')).withColumn('ano_mes',F.when(F.col('mes').between(1,12),F.col('ano')*100+F.col('mes')))
 d=d.withColumn('cid_codigo',F.regexp_replace(F.upper(F.col(diagnostico).cast('string')),r'[^A-Z0-9]',''))
 return d.withColumn('cardio',F.when(F.col('cid_codigo').rlike(r'^I[0-9]{2}'),1).otherwise(0))
sih_geo=prep(sih,'municipio_residencia','cid_principal',F.col('ano_competencia'),F.col('mes_competencia'))
sim_geo=prep(sim,'municipio_residencia','cid_causa_basica',F.year('data_obito'),F.month('data_obito'))
gold_dim_cid=sih_geo.select('cid_codigo').unionByName(sim_geo.select('cid_codigo')).filter(F.col('cid_codigo').rlike(r'^[A-Z][0-9]{2}')).distinct().withColumn('cid_categoria',F.substring('cid_codigo',1,3)).withColumn('doenca_cardiovascular',F.col('cid_codigo').rlike(r'^I[0-9]{2}'))
periodos=sih_geo.select('ano','mes').unionByName(sim_geo.select('ano','mes'))
for c in (st,lt):
 if c is not None:periodos=periodos.unionByName(c.select(F.col('ano_arquivo').cast('int').alias('ano'),F.col('mes_arquivo').cast('int').alias('mes')))
gold_dim_tempo=periodos.filter(F.col('mes').between(1,12)).distinct().withColumn('ano_mes',F.col('ano')*100+F.col('mes')).withColumn('data_inicio_mes',F.make_date('ano','mes',F.lit(1))).withColumn('trimestre',F.quarter('data_inicio_mes'))
TABELAS_GOLD={GOLD['CID']:gold_dim_cid,GOLD['TEMPO']:gold_dim_tempo}


In [ ]:
gold_fato_internacoes=sih_geo.groupBy('municipio_residencia_codigo','codigo_uf_ibge','uf_residencia','ano','mes','ano_mes','cid_codigo').agg(F.count('*').alias('total_internacoes'),F.sum('cardio').alias('internacoes_cardiovasculares'),F.sum('valor_total').alias('valor_total'),F.sum('valor_uti').alias('valor_uti'),F.sum('dias_permanencia').alias('dias_permanencia'),F.sum(F.when(F.col('indicador_obito')==1,1).otherwise(0)).alias('obitos_hospitalares'))
gold_fato_mortalidade=sim_geo.groupBy('municipio_residencia_codigo','codigo_uf_ibge','uf_residencia','ano','mes','ano_mes','cid_codigo','sexo_codigo').agg(F.count('*').alias('total_obitos'),F.sum('cardio').alias('obitos_cardiovasculares'))
TABELAS_GOLD[GOLD['INTERNACOES']]=gold_fato_internacoes
TABELAS_GOLD[GOLD['MORTALIDADE']]=gold_fato_mortalidade
if st is not None:
 gold_fato_cnes_estabelecimentos=st.groupBy('uf_arquivo','ano_arquivo','mes_arquivo','municipio_codigo_6','cnes_estabelecimento','tipo_unidade','tipo_gestao').agg(F.count('*').alias('registros_cadastrais'))
 TABELAS_GOLD[GOLD['ESTABELECIMENTOS']]=gold_fato_cnes_estabelecimentos
if lt is not None:
 gold_fato_cnes_leitos=lt.groupBy('uf_arquivo','ano_arquivo','mes_arquivo','municipio_codigo_6','cnes_estabelecimento','codigo_leito','tipo_leito').agg(F.sum('quantidade_existente').alias('leitos_existentes'),F.sum('quantidade_sus').alias('leitos_sus'),F.count('*').alias('registros_leitos'))
 TABELAS_GOLD[GOLD['LEITOS']]=gold_fato_cnes_leitos
def cobertura(df,nome,anual=False):
 g=[F.col('uf_arquivo').alias('uf'),F.col('ano_arquivo').alias('ano')]
 if not anual:g.append(F.col('mes_arquivo').alias('mes'))
 d=df.groupBy(*g).agg(F.count('*').alias('registros'),F.countDistinct('arquivo_id').alias('arquivos'))
 if anual:d=d.withColumn('mes',F.lit(None).cast('int'))
 return d.withColumn('fonte',F.lit(nome)).select('fonte','uf','ano','mes','registros','arquivos')
gold_controle_cobertura=cobertura(sih,'SIH').unionByName(cobertura(sim,'SIM',True))
for nome,df in [('CNES_ST',st),('CNES_LT',lt)]:
 if df is not None:gold_controle_cobertura=gold_controle_cobertura.unionByName(cobertura(df,nome))
TABELAS_GOLD[GOLD['COBERTURA']]=gold_controle_cobertura
a=sih_geo.filter(F.col('uf_residencia').isNotNull()).groupBy(F.col('uf_residencia').alias('uf'),'ano').agg(F.count('*').alias('internacoes'),F.sum('cardio').alias('internacoes_cardiovasculares'),F.sum('valor_total').alias('valor_internacoes'),F.countDistinct('mes').alias('meses_sih_carregados'))
b=sim_geo.filter(F.col('uf_residencia').isNotNull()).groupBy(F.col('uf_residencia').alias('uf'),'ano').agg(F.count('*').alias('obitos'),F.sum('cardio').alias('obitos_cardiovasculares'),F.countDistinct('mes').alias('meses_com_obitos'))
TABELAS_GOLD[GOLD['INDICADORES']]=a.join(b,['uf','ano'],'full_outer')


In [ ]:
totais=[('SIH',sih.count(),gold_fato_internacoes.agg(F.sum('total_internacoes')).first()[0]),('SIM',sim.count(),gold_fato_mortalidade.agg(F.sum('total_obitos')).first()[0])]
if st is not None:totais.append(('CNES_ST',st.count(),gold_fato_cnes_estabelecimentos.agg(F.sum('registros_cadastrais')).first()[0]))
if lt is not None:totais.append(('CNES_LT',lt.count(),gold_fato_cnes_leitos.agg(F.sum('registros_leitos')).first()[0]))
for n,s,g in totais:
 print('RECONCILIACAO',n,s,int(g or 0))
 assert s==int(g or 0),f'{n}: divergencia quantitativa'
if PUBLICAR_GOLD:
 if not CONFIRMO_LAKEHOUSE_PADRAO:raise RuntimeError('Confirme que lh_media_analytics e o Lakehouse padrao')
 # A versao Gold utiliza overwrite: nao rode enquanto o NB05 estiver gravando na Silver.
 # Para estabilidade de schema no Direct Lake, valide o modelo semantico apos cada atualizacao.
 for tabela,df in TABELAS_GOLD.items():
  print('PUBLICANDO',tabela)
  if tem(tabela):df.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(tabela)
  else:df.write.format('delta').mode('errorifexists').saveAsTable(tabela)
  spark.catalog.refreshTable(tabela)
else:print('SIMULACAO, SEM PUBLICACAO')
